In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Jalankan circuit pertamamu di perangkat keras

<Accordion>
<AccordionItem title="Versi paket">

Kode di halaman ini dikembangkan menggunakan persyaratan berikut.
Kami sarankan menggunakan versi ini atau yang lebih baru.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Contoh ini terdiri dari dua bagian. Pertama, kamu akan membuat program kuantum "Hello world" sederhana dan menjalankannya di unit pemrosesan kuantum (QPU). Karena penelitian kuantum nyata membutuhkan program yang jauh lebih kuat, di bagian kedua ([Skalakan ke jumlah qubit yang besar](#scale-to-large-numbers-of-qubits)), kamu akan menskalakan program sederhana tersebut ke tingkat utilitas.

## Instal dan autentikasi
1. Jika kamu belum menginstal Qiskit, temukan petunjuknya di panduan [Quickstart](/guides/quick-start).

    - Instal Qiskit Runtime untuk menjalankan pekerjaan pada perangkat keras kuantum:

        ```bash
        pip install qiskit-ibm-runtime
        ```

    - Siapkan lingkungan untuk menjalankan notebook Jupyter secara lokal:

        ```bash
        pip install jupyter
        ```

2. Atur autentikasi untuk akses ke perangkat keras kuantum melalui [Open Plan](/guides/plans-overview#open-plan) yang gratis.

    (Jika kamu mendapat undangan melalui email untuk bergabung ke akun, ikuti [langkah untuk pengguna yang diundang](/guides/cloud-setup-invited) sebagai gantinya.)

    - Buka [IBM Quantum Platform](https://quantum.cloud.ibm.com/) untuk login atau membuat akun.
         > **Note:** Jika kamu terhubung melalui server proxy, kamu harus menggunakan Qiskit Runtime v0.44.0 atau lebih baru.
    - Buat API key-mu (juga disebut *API token*) di [dashboard](https://quantum.cloud.ibm.com/), lalu salin ke lokasi yang aman.
    - Buka halaman [Instances](https://quantum.cloud.ibm.com/instances) dan temukan instance yang ingin kamu gunakan. Arahkan kursor ke CRN-nya dan klik untuk menyalinnya.

    - Simpan kredensialmu secara lokal dengan kode ini:

        ```python
        from qiskit_ibm_runtime import QiskitRuntimeService

        QiskitRuntimeService.save_account(
        token="<your-api-key>", # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
        instance="<CRN>", # Optional
        )
        ```

3. Sekarang kamu bisa menggunakan kode Python ini setiap kali ingin autentikasi ke Qiskit Runtime Service:
    ```python
    from qiskit_ibm_runtime import QiskitRuntimeService

    # Run every time you need the service
    service = QiskitRuntimeService()
    ```
> **Info:** Jika kamu menggunakan komputer umum atau lingkungan yang tidak aman, ikuti [petunjuk autentikasi manual](/guides/cloud-setup-untrusted) sebagai gantinya untuk menjaga keamanan kredensial autentikasimu.
## Buat dan jalankan program kuantum sederhana
Empat langkah dalam menulis program kuantum menggunakan pola Qiskit adalah:

1.  Petakan masalah ke format native kuantum.

2.  Optimalkan Circuit dan operator.

3.  Eksekusi menggunakan fungsi primitif kuantum.

4.  Analisis hasilnya.

### Langkah 1. Petakan masalah ke format native kuantum
Dalam program kuantum, *Circuit kuantum* adalah format native untuk merepresentasikan instruksi kuantum, dan *operator* merepresentasikan observabel yang akan diukur. Saat membuat Circuit, biasanya kamu akan membuat objek [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#quantumcircuit-class) baru, lalu menambahkan instruksi secara berurutan.
Sel kode berikut membuat Circuit yang menghasilkan *Bell state*, yaitu keadaan di mana dua qubit saling terjalin sepenuhnya.

> **Note:** Qiskit SDK menggunakan penomoran bit LSb 0 di mana digit ke-$n$ memiliki nilai $1 \ll n$ atau $2^n$. Untuk detail lebih lanjut, lihat topik [Bit-ordering in the Qiskit SDK](/guides/bit-ordering).

In [1]:
# This cell is hidden from users.  It hides several unnecessary warnings.

import warnings
import logging

warnings.filterwarnings("ignore", message=".*Instance was not set*")
warnings.filterwarnings("ignore", message=".*loading instance*")
warnings.filterwarnings("ignore", message=".*using instance*")

# This cell is hidden from users.  It hides several unnecessary warnings.


class IgnoreSpecificMessages(logging.Filter):
    def filter(self, record):
        for text in [
            "Instance was not set",
            "Loading default saved account",
            "Loading instance",
            "Instance was not set",
            "using instance",
        ]:
            if text in record.getMessage():
                return False
        return True


logger = logging.getLogger("qiskit_ibm_runtime")
logger.addFilter(IgnoreSpecificMessages())

for handler in logger.handlers:
    handler.addFilter(IgnoreSpecificMessages())

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorOptions
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from matplotlib import pyplot as plt
# Uncomment the next line if you want to use a simulator:
# from qiskit_ibm_runtime.fake_provider import FakeBelemV2


# Create a new circuit with two qubits
qc = QuantumCircuit(2)

# Add a Hadamard gate to qubit 0
qc.h(0)

# Perform a controlled-X gate on qubit 1, controlled by qubit 0
qc.cx(0, 1)

# Return a drawing of the circuit using MatPlotLib ("mpl").
# These guides are written by using Jupyter notebooks, which
# display the output of the last line of each cell.
# If you're running this in a script, use `print(qc.draw())` to
# print a text drawing.
qc.draw("mpl")

<Image src="../docs/images/guides/hello-world/extracted-outputs/930ca3b6-0.svg" alt="Output of the previous code cell" />

> **Note:** Di sini, operator seperti `ZZ` adalah singkatan dari produk tensor $Z\otimes Z$, yang berarti mengukur Z pada qubit 1 dan Z pada qubit 0 secara bersamaan, dan mendapatkan informasi tentang korelasi antara qubit 1 dan qubit 0. Nilai ekspektasi seperti ini juga biasanya ditulis sebagai $\langle Z_1 Z_0 \rangle$.
> 
> Jika keadaannya terjalin, maka pengukuran $\langle Z_1 Z_0 \rangle$ seharusnya berbeda dari pengukuran $\langle I_1 \otimes Z_0 \rangle \langle Z_1 \otimes I_0 \rangle$. Untuk keadaan terjalin spesifik yang dibuat oleh Circuit yang dijelaskan di atas, pengukuran $\langle Z_1 Z_0 \rangle$ seharusnya 1 dan pengukuran $\langle I_1 \otimes Z_0 \rangle \langle Z_1 \otimes I_0 \rangle$ seharusnya nol.
<span id="optimize"></span>

### Langkah 2. Optimalkan Circuit dan operator

Saat mengeksekusi Circuit pada perangkat, penting untuk mengoptimalkan sekumpulan instruksi yang ada dalam Circuit dan meminimalkan kedalaman keseluruhan (kurang lebih jumlah instruksi) dari Circuit. Ini memastikan kamu mendapatkan hasil terbaik dengan mengurangi efek error dan noise. Selain itu, instruksi Circuit harus sesuai dengan [Instruction Set Architecture (ISA)](/guides/transpile#instruction-set-architecture) perangkat Backend dan harus mempertimbangkan gate dasar serta konektivitas qubit perangkat.

Kode berikut menginstansiasi perangkat nyata untuk mengirimkan pekerjaan dan mengubah Circuit serta observabel agar sesuai dengan ISA Backend tersebut. Ini memerlukan bahwa kamu sudah [menyimpan kredensialmu](/guides/cloud-setup).

In [3]:
# Set up six different observables.

observables_labels = ["IZ", "IX", "ZI", "XI", "ZZ", "XX"]
observables = [SparsePauliOp(label) for label in observables_labels]

![Output of the previous code cell](../docs/images/guides/hello-world/extracted-outputs/9a901271-0.svg)

### Langkah 3. Eksekusi menggunakan primitif kuantum

Komputer kuantum bisa menghasilkan hasil acak, jadi biasanya kamu mengumpulkan sampel dari output dengan menjalankan Circuit berkali-kali. Kamu bisa memperkirakan nilai observabel menggunakan kelas `Estimator`. `Estimator` adalah salah satu dari dua [primitif](/guides/get-started-with-estimator); yang lainnya adalah `Sampler`, yang bisa digunakan untuk mendapatkan data dari komputer kuantum. Objek-objek ini memiliki metode `run()` yang mengeksekusi pilihan Circuit, observabel, dan parameter (jika berlaku), menggunakan [primitive unified bloc (PUB)](/guides/get-started-with-sampler).

In [4]:
service = QiskitRuntimeService()

backend = service.least_busy(simulator=False, operational=True)

# Convert to an ISA circuit and layout-mapped observables.
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

isa_circuit.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/hello-world/extracted-outputs/9a901271-0.svg" alt="Output of the previous code cell" />

### Step 3. Execute using the quantum primitives

Quantum computers can produce random results, so you usually collect a sample of the outputs by running the circuit many times. You can estimate the value of the observable by using the `Estimator` class. `Estimator` is one of two [primitives](/docs/guides/get-started-with-estimator); the other is `Sampler`, which can be used to get data from a quantum computer.  These objects possess a `run()` method that executes the selection of circuits, observables, and parameters (if applicable), using a [primitive unified bloc (PUB)](/docs/guides/get-started-with-sampler).

In [5]:
# Construct the Estimator instance.

estimator = Estimator(mode=backend)
estimator.options.resilience_level = 1
estimator.options.default_shots = 5000

mapped_observables = [
    observable.apply_layout(isa_circuit.layout) for observable in observables
]

# One pub, with one circuit to run against five different observables.
job = estimator.run([(isa_circuit, mapped_observables)])

# Use the job ID to retrieve your job data later
print(f">>> Job ID: {job.job_id()}")

>>> Job ID: d8286mfoha1c73bl8hrg


Setelah pekerjaan dikirim, kamu bisa menunggu hingga pekerjaan selesai dalam instance Python saat ini, atau menggunakan `job_id` untuk mengambil data di lain waktu. (Lihat [bagian tentang mengambil pekerjaan](/guides/monitor-job#retrieve-job-results-at-a-later-time) untuk detailnya.)

Setelah pekerjaan selesai, periksa outputnya melalui atribut `result()` dari pekerjaan.

In [6]:
# This is the result of the entire submission.  You submitted one Pub,
# so this contains one inner result (and some metadata of its own).
job_result = job.result()

# This is the result from our single pub, which had six observables,
# so contains information on all six.
pub_result = job.result()[0]

In [7]:
# Check there are six observables.
# If not, edit the comments in the previous cell and update this test.
assert len(pub_result.data.evs) == 6

:::note[Alternatif: jalankan contoh menggunakan simulator]

  Saat kamu menjalankan program kuantummu di perangkat nyata, beban kerjamu harus menunggu dalam antrian sebelum dijalankan. Untuk menghemat waktu, kamu bisa menggunakan kode berikut untuk menjalankan beban kerja kecil ini di [`fake_provider`](../api/qiskit-ibm-runtime/fake-provider) dengan mode pengujian lokal Qiskit Runtime. Perhatikan bahwa ini hanya memungkinkan untuk Circuit kecil. Saat kamu melakukan skalasi di bagian berikutnya, kamu perlu menggunakan perangkat nyata.

In [8]:
# Plot the result

values = pub_result.data.evs

errors = pub_result.data.stds

# plotting graph
plt.plot(observables_labels, values, "-o")
plt.xlabel("Observables")
plt.ylabel("Values")
plt.show()

<Image src="../docs/images/guides/hello-world/extracted-outputs/87143fcc-0.svg" alt="Output of the previous code cell" />

:::
### Langkah 4. Analisis hasilnya
Langkah analisis biasanya adalah tempat kamu bisa memproses hasilmu menggunakan, misalnya, mitigasi error pengukuran atau zero noise extrapolation (ZNE). Kamu bisa memasukkan hasil ini ke dalam alur kerja lain untuk analisis lebih lanjut atau menyiapkan plot nilai dan data utama. Secara umum, langkah ini spesifik untuk masalahmu. Untuk contoh ini, buat plot setiap nilai ekspektasi yang diukur untuk Circuit kita.

Nilai ekspektasi dan standar deviasi untuk observabel yang kamu tentukan ke Estimator diakses melalui atribut `PubResult.data.evs` dan `PubResult.data.stds` dari hasil pekerjaan. Untuk mendapatkan hasil dari Sampler, gunakan fungsi `PubResult.data.meas.get_counts()`, yang akan mengembalikan `dict` pengukuran dalam bentuk bitstring sebagai kunci dan hitungan sebagai nilai yang sesuai. Untuk informasi lebih lanjut, lihat panduan [Sampler quickstart](/guides/get-started-with-sampler).

  ```python

  # Use the following code instead if you want to run on a simulator:

  from qiskit_ibm_runtime.fake_provider import FakeBelemV2
  backend = FakeBelemV2()
  estimator = Estimator(backend)

  # Convert to an ISA circuit and layout-mapped observables.

  pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
  isa_circuit = pm.run(qc)
  mapped_observables = [
  observable.apply_layout(isa_circuit.layout) for observable in observables
  ]

  job = estimator.run([(isa_circuit, mapped_observables)])
  result = job.result()

  # This is the result of the entire submission.  You submitted one Pub,
  # so this contains one inner result (and some metadata of its own).

  job_result = job.result()

  # This is the result from our single pub, which had five observables,
  # so contains information on all five.

  pub_result = job.result()[0]
  ```

![Output of the previous code cell](../docs/images/guides/hello-world/extracted-outputs/87143fcc-0.svg)

Perhatikan bahwa untuk qubit 0 dan 1, nilai ekspektasi independen dari X dan Z keduanya adalah 0, sedangkan korelasi (`XX` dan `ZZ`) adalah 1. Ini adalah ciri khas keterikatan kuantum.

In [9]:
# Make sure the results follow the claim from the previous markdown cell.
# This can happen when the device occasionally behaves strangely. If this cell
# fails, you may just need to run the notebook again.

_results = {obs: val for obs, val in zip(observables_labels, values)}
for _label in ["IZ", "IX", "ZI", "XI"]:
    assert abs(_results[_label]) < 0.2
for _label in ["XX", "ZZ"]:
    assert _results[_label] > 0.8

## Skalakan ke jumlah qubit yang besar
Dalam komputasi kuantum, pekerjaan skala utilitas sangat penting untuk kemajuan di bidang ini. Pekerjaan semacam itu membutuhkan komputasi dalam skala yang jauh lebih besar; bekerja dengan Circuit yang mungkin menggunakan lebih dari 100 qubit dan lebih dari 1000 Gate. Contoh ini menunjukkan bagaimana kamu bisa menyelesaikan pekerjaan skala utilitas di IBM&reg; QPU dengan membuat dan menganalisis keadaan GHZ 100-qubit. Ini menggunakan alur kerja pola Qiskit dan diakhiri dengan mengukur nilai ekspektasi $\langle Z_0 Z_i \rangle$ untuk setiap qubit.

### Langkah 1. Petakan masalahnya
Tulis fungsi yang mengembalikan `QuantumCircuit` yang mempersiapkan keadaan GHZ $n$-qubit (pada dasarnya Bell state yang diperluas), lalu gunakan fungsi tersebut untuk mempersiapkan keadaan GHZ 100-qubit dan kumpulkan observabel yang akan diukur.

In [10]:
def get_qc_for_n_qubit_GHZ_state(n: int) -> QuantumCircuit:
    """This function will create a qiskit.QuantumCircuit (qc) for an n-qubit GHZ state.

    Args:
        n (int): Number of qubits in the n-qubit GHZ state

    Returns:
        QuantumCircuit: Quantum circuit that generate the n-qubit GHZ state, assuming all qubits start in the 0 state
    """
    if isinstance(n, int) and n >= 2:
        qc = QuantumCircuit(n)
        qc.h(0)
        for i in range(n - 1):
            qc.cx(i, i + 1)
    else:
        raise Exception("n is not a valid input")
    return qc


# Create a new circuit with 100 qubits in the GHZ state
n = 100
qc = get_qc_for_n_qubit_GHZ_state(n)

Selanjutnya, petakan ke operator yang diminati. Contoh ini menggunakan operator `ZZ` antara qubit untuk memeriksa perilaku saat qubit semakin jauh terpisah. Nilai ekspektasi antara qubit yang jauh yang semakin tidak akurat (rusak) akan mengungkapkan tingkat noise yang ada.

In [11]:
# ZZII...II, ZIZI...II, ... , ZIII...IZ
operator_strings = [
    "Z" + "I" * i + "Z" + "I" * (n - 2 - i) for i in range(n - 1)
]

operators = [SparsePauliOp(operator) for operator in operator_strings]

### Step 2. Optimize the problem for execution on quantum hardware

The following code transforms the circuit and observables to match the backend's ISA. It requires that you have already [saved your credentials](/docs/guides/cloud-setup)

In [12]:
service = QiskitRuntimeService()

backend = service.least_busy(
    simulator=False, operational=True, min_num_qubits=100
)
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)

isa_circuit = pm.run(qc)
isa_operators_list = [op.apply_layout(isa_circuit.layout) for op in operators]

### Langkah 2. Optimalkan masalah untuk eksekusi pada perangkat keras kuantum
Kode berikut mengubah Circuit dan observabel agar sesuai dengan ISA Backend. Ini memerlukan bahwa kamu sudah [menyimpan kredensialmu](/guides/cloud-setup).

In [13]:
options = EstimatorOptions()
options.resilience_level = 1
options.dynamical_decoupling.enable = True
options.dynamical_decoupling.sequence_type = "XY4"

# Create an Estimator object
estimator = Estimator(backend, options=options)

In [14]:
# Submit the circuit to Estimator
job = estimator.run([(isa_circuit, isa_operators_list)])
job_id = job.job_id()
print(job_id)

d828aeo0bvlc73d1vs20


### Step 4. Post-process results

After the job completes, plot the results and notice that $\langle Z_0 Z_i \rangle$ decreases with increasing $i$, even though in an ideal simulation all $\langle Z_0 Z_i \rangle$ should be 1.

In [15]:
# data
data = list(range(1, len(operators) + 1))  # Distance between the Z operators
result = job.result()[0]
values = result.data.evs  # Expectation value at each Z operator.
values = [
    v / values[0] for v in values
]  # Normalize the expectation values to evaluate how they decay with distance.

# plotting graph
plt.plot(data, values, marker="o", label="100-qubit GHZ state")
plt.xlabel("Distance between qubits $i$")
plt.ylabel(r"$\langle Z_i Z_0 \rangle / \langle Z_1 Z_0 \rangle $")
plt.legend()
plt.show()

<Image src="../docs/images/guides/hello-world/extracted-outputs/de91ebd0-0.svg" alt="Output of the previous code cell" />

The previous plot shows that as the distance between qubits increases, the signal decays because of the presence of noise.

## Next steps

<Admonition type="tip" title="Recommendations">
  -   Try one of these tutorials:
      - [Ground-state energy estimation of the Heisenberg chain with VQE](/docs/tutorials/spin-chain-vqe)
      - Solve optimization problems using [QAOA](/docs/tutorials/quantum-approximate-optimization-algorithm)
      - Train [quantum kernel](/docs/tutorials/quantum-kernel-training) models for machine learning tasks
  - Find detailed installation instructions in the [Install Qiskit](/docs/guides/install-qiskit) guide.
  - If you prefer not to install Qiskit locally, read about options to use Qiskit in an [online development environment](/docs/guides/online-lab-environments).
  - To save multiple account credentials or to specify other account options, see detailed instructions in the [Save your login credentials](/docs/guides/save-credentials#save-your-access-credentials) guide.
</Admonition>